# TuRBO-1 入門：一つの探索領域を育てるベイズ最適化

**このNotebookのゴール**：GPによる予測、次の実験候補の提案、実測値による状態更新を、自分で順に実行できるようになることです。TuRBO-1は一つの局所探索領域（Trust Region）を持ち、改善が続くと広げ、改善が止まると狭めます。

**想定する読者**：Pythonの基本操作を知っている方。GPやTuRBOの事前知識は不要です。

**実行方法**：Python 3.13以降のrobotorchan環境で、上から順にセルを実行してください。最適化のモデル学習には少し時間がかかります。以下のコードセルは既存の実行例を維持しています。

**最初に覚える記号**：`X` は入力条件、`Y` は実測した品質、`d` は入力変数の数、`q` は一度に試す候補数、`center` は探索の中心、`length` は領域の相対的な幅です。

**全体の流れ**：初期実験 → GPを学習 → 局所範囲を作る → 候補を選ぶ → 実験する → 成功・失敗を判定 → 次の反復。予測値だけでは状態を更新しません。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import SingleTaskGP
from robotorchan.optim import TuRBOState, TuRBOStrategy

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 例題の意味

ここでは温度・圧力に相当する二つの入力を、どちらも0〜1に正規化して扱います。目的関数は「目標条件からの距離の二乗にマイナスを付けた値」です。**0に近いほど良い品質**で、最大値は0です。実務では、この関数の代わりに実験装置やシミュレーターから観測値を取得します。

最初の10点は学習用の実測データに相当します。ここで求める`initial_center`は、最初の10点の中で最も良かった条件です。

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    optimum = torch.tensor([0.72, 0.28])
    return -((X - optimum) ** 2).sum(dim=-1, keepdim=True)


dim = 2
bounds = torch.stack([torch.zeros(dim), torch.ones(dim)])
train_X = torch.rand(10, dim)
train_Y = objective(train_X)
best_index = train_Y.squeeze(-1).argmax()
initial_center = train_X[best_index].clone()
initial_best = float(train_Y[best_index].item())

print("initial center:", initial_center)
print("initial best:", initial_best)

### 状態と探索範囲を分けて考える

`TuRBOState`は領域長、成功・失敗回数、最良値などを記憶します。`TuRBOStrategy`は中心座標と状態を使い、実際の探索範囲を作ります。**この段階では新しい実験を行いません。**

例えば全体範囲が0〜1、中心が0.5、領域長が0.8、方向の重みが1なら、局所範囲はおおむね0.1〜0.9です。中心が境界に近い場合は全体範囲で切り詰められます。下の出力の`local bounds`を確認してください。

In [ ]:
state = TuRBOState(
    dim=dim,
    batch_size=1,
    best_value=initial_best,
)
strategy = TuRBOStrategy(
    bounds,
    center=initial_center,
    state=state,
    num_restarts=3,
    raw_samples=64,
    seed=11,
)
local_bounds = strategy.trust_region_bounds()

assert local_bounds.shape == bounds.shape
assert torch.all(local_bounds >= bounds[0])
assert torch.all(local_bounds <= bounds[1])
print("trust-region length:", strategy.state.length)
print("local bounds:\n", local_bounds)

### GPと獲得関数の役割

`SingleTaskGP`は観測済みの`train_X`と`train_Y`から、未観測点の品質を予測します。`qLogExpectedImprovement`は「これまでの最高値を改善できそうか」を候補の価値として評価します。`strategy.optimize`は**全体範囲ではなく現在の局所範囲**で候補を選びます。

`candidate`は**予測に基づいて選ばれた未評価の入力**です。まだ品質の実測値ではありません。

In [ ]:
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()
acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
)
result = strategy.optimize(acquisition, q=1)
candidate = result.candidates.detach()
candidate_bounds = result.metadata["trust_region_bounds"]

assert candidate.shape == torch.Size([1, dim])
assert torch.all(candidate >= candidate_bounds[0])
assert torch.all(candidate <= candidate_bounds[1])
print("candidate:", candidate)
print("candidate-generation:", result.metadata["candidate_generation"])

### ここで初めて実験を行う

`objective(candidate)`が実際の評価に相当します。その結果`new_Y`を受け取ってから`update_state`を呼びます。新しい結果が十分に良ければ成功カウンタが増え、そうでなければ失敗カウンタが増えます。成功・失敗が一定回数続くと領域長が変わります。

**観察ポイント**：`old_best`と新しい最良値、`old_center`と更新後の中心、二つのカウンタを比較してください。1回の実験で必ず領域長が変わるわけではありません。

In [ ]:
new_Y = objective(candidate)
old_center = strategy.center.clone()
old_best = strategy.state.best_value
strategy.update_state(new_Y, candidates=candidate)

train_X = torch.cat([train_X, candidate], dim=0)
train_Y = torch.cat([train_Y, new_Y], dim=0)

print("observed value:", new_Y.item())
print("best value:", old_best, "->", strategy.state.best_value)
print("center:", old_center, "->", strategy.center)
print("success counter:", strategy.state.success_counter)
print("failure counter:", strategy.state.failure_counter)

### 4反復の流れを追う

ここからは同じ処理を4回繰り返します。毎回、**最新の観測データでGPを学習し直す**ことが重要です。表示される各行は、順に「反復番号、領域長、状態の最良値、成功回数、失敗回数」です。

初期点やGPの予測によって、4回では領域の拡大・縮小が起きない場合もあります。これは異常ではありません。領域の変化には所定回数の連続した成功・失敗が必要です。

In [ ]:
history = []
for iteration in range(4):
    model = SingleTaskGP(train_X, train_Y)
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    acquisition = qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
    )
    result = strategy.optimize(acquisition, q=1)
    new_X = result.candidates.detach()
    new_Y = objective(new_X)
    strategy.update_state(new_Y, candidates=new_X)
    train_X = torch.cat([train_X, new_X], dim=0)
    train_Y = torch.cat([train_Y, new_Y], dim=0)
    history.append(
        (
            iteration,
            strategy.state.length,
            strategy.state.best_value,
            strategy.state.success_counter,
            strategy.state.failure_counter,
        )
    )

for row in history:
    print(row)

### 別の候補選択法：Thompson Sampling

獲得関数を最大化する代わりに、GPの事後分布から「あり得る品質関数」を抽選し、その関数上で良い候補を選ぶ方法です。**予測平均だけで決めない**ため、不確実な場所を選ぶこともあります。

下のセルは候補を生成し、品質を計算して表示する例です。この結果は学習データやTuRBO状態へ追加していません。次の反復へ反映する場合は、実測値を使った状態更新とデータ追加が必要です。

In [ ]:
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()
ts_result = strategy.thompson_sample(
    model,
    q=1,
    n_candidates=128,
)
ts_candidate = ts_result.candidates
ts_bounds = ts_result.metadata["trust_region_bounds"]

assert ts_candidate.shape == torch.Size([1, dim])
assert torch.all(ts_candidate >= ts_bounds[0])
assert torch.all(ts_candidate <= ts_bounds[1])
assert ts_result.metadata["candidate_generation"] == "thompson"
print("Thompson candidate:", ts_candidate)

# Thompson候補も元の目的関数で評価します。
ts_observed_Y = objective(ts_candidate.detach())
assert ts_observed_Y.shape == torch.Size([1, 1])
print("Thompson candidate objective:", ts_observed_Y)


## 8. Trust-region state is not optimizer state

TuRBO stateが保持するのはincumbent、trust-region length、success/failure counters、restart情報です。AdamやL-BFGS等のnumerical optimizer stateとは別物です。

```text
TuRBO strategy = where to search
acquisition optimizer = how to optimize inside that region
surrogate model = what posterior to believe
acquisition = what candidate utility to optimize
```

### Restartは「やり直し」ではなく「探索場所の変更」

改善が得られず領域が小さくなりすぎた場合は、別の中心から再探索できます。Restartしても、過去に観測した最高値をなかったことにする必要はありません。一方、**GPの学習データをどう扱うかは別の設計判断**です。

## 10. High-dimensional and reduced models

TuRBOは既定でpublic/original input spaceを局所化します。PCA/PLS/RandomProjection modelが内部latent representationを使っても、candidateとtrust-region geometryを暗黙にlatentへ移しません。

public dimensionと1対1対応するARD lengthscaleがある場合はdimension weightsへ反映できます。一方、latent GP lengthscaleをraw-space geometryへ暗黙逆写像しません。

## 11. Async, robust, mixed, and multi-objective boundaries

- **Async**: unresolved evaluationsはacquisitionの`X_pending`。TuRBO stateはcompleted resultsだけで更新します。
- **Fantasization**: model/acquisition responsibility。fantasy batchはTuRBO `batch_size`ではありません。
- **Robust BO**: input perturbation scenario axisはtrust-region dimensionへ追加しません。必要なら`state_values`にrobust utilityを渡します。
- **Mixed**: continuous dimensionsだけをlengthで局所化し、integer/categorical合法値集合へ人工的な数値距離を導入しません。
- **Multi-objective**: qLogEHVI/qLogNEHVI等はlocal regionで最適化できますが、TuRBO-1 state updateにはcandidateごとのscalar state utilityが必要です。
- **Candidate constraints**: trust regionは既知input constraintsの代替ではありません。両方を同時に満たす必要があります。

### 学習の振り返り

1. GPの予測値と`objective`による実測値は、どこで区別されていましたか？
2. `length`を小さくすると、候補を探す範囲はどう変わりますか？
3. `q=1`のこの例で、成功・失敗は何点の実測結果を使って判定していますか？
4. Thompson Samplingで生成した候補は、自動的に学習データへ追加されますか？

**答え**：1. 実測は`objective`を呼んだ後です。2. 中心付近の狭い範囲になります。3. 1点です。4. いいえ、明示的な追加が必要です。

**次に読む**：[TuRBO-1とTuRBO-mの理論](../../docs/theory/optimization/10_trust_region.md) ／ [TuRBO-m Notebook](43_turbo_multi_benchmark.ipynb)。